### Imports

In [ ]:
import os
import sys
import yaml
import sglang as sgl
from sglang.lang.chat_template import get_chat_template
from datetime import datetime
import multiprocessing as mp



# Add the project root directory to sys.path
project_root = os.path.abspath('..')  # Adjust this path if needed
sys.path.append(project_root)

from utils import load_images_and_combine
from pipeline_generate import build_pred_csv_sgl

### Datasets


In [ ]:
# Load the YAML file into a dictionary
dataset_yaml_file = '/workspace/home/data/datasets.yaml'
with open(dataset_yaml_file, 'r') as file:
    datasets = yaml.safe_load(file)

#datasets

# Genereate outputs using sgl on the ISIC 2019 dataset
## Load data

In [ ]:
# get dataset
dataset_key = 'ISIC2019'
dataset = datasets[dataset_key]

# load the ground truth, predicted segmentations and images
images_path = f"/workspace/home/data/{datasets[dataset_key]['dataset_dir']}/raw"

images = load_images_and_combine(dir_path=images_path, confounder_dir=dataset['confounder_dir'], no_confounder_dir=dataset['no_confounder_dir'])

## Setup SGL 

### Start the SGL server
Run the following cell to generate the command line to start the sgl server

In [ ]:
model_path = "liuhaotian/llava-v1.6-vicuna-7b" # can I get it from backend?
tokenizer_path = "llava-hf/llava-1.5-7b-hf"
chat_template = "vicuna_v1.1"


#model_path="lmms-lab/llama3-llava-next-8b",
#tokenizer_path="lmms-lab/llama3-llava-next-8b-tokenizer",
#chat_template = "llama-3-instruct"

mp.set_start_method("spawn", force=True)
runtime = sgl.Runtime(
        model_path=model_path,
        tokenizer_path=tokenizer_path,
    )
runtime.endpoint.chat_template = get_chat_template(chat_template)
sgl.set_default_backend(runtime)

To manually start the server run command giving by this string in the terminal:

```sgl_client_string = f"python3 -m sglang.launch_server --model-path {model_path} --tokenizer-path {tokenizer_path} --chat-template {chat_template} --port {port}"```

Dont forget to set the port and start the runtime endpoint as follows:

```sgl.set_runtime_endpoint(f"http://localhost:{port}")```


### Setup output directory

In [ ]:
# Generate a timestamp
timestamp = datetime.now().strftime("%Y%m%dT%H%M%S")

output_dir = f"/workspace/home/pipeline_outputs/{datasets[dataset_key]['dataset_dir']}/{timestamp}"
os.makedirs(output_dir, exist_ok=True)


### Inference
The ```user_input_list``` and ```output_for_true``` can be set in the datasets.yaml file. The ```user_input_list``` is a list of the questions feeded to the model in the given order. The ```output_for_true``` are the answers of the model that we expect if the confounder is present and the label is set to true.

In [ ]:
# csv file path
csv_file = f"{output_dir}/sgl_pred.csv"

# Questions for (multi-turn) chat with model 
user_input_list = dataset['user_input_list']

# Model outputs that are considered as True 
output_for_true = dataset['output_for_true']

build_pred_csv_sgl(csv_file, images, user_input_list, output_for_true)

### Shutdown sgl server

In [ ]:
runtime.shutdown()

### Generate Metadata YAML

In [ ]:
# Generate meta data yaml file
meta_data = {
    'timestamp': timestamp,
    'dataset': dataset_key,
    'model': model_path,
    'tokenizer': tokenizer_path,
    'chat_template': chat_template,
    'user_input_list': user_input_list,
    'output_for_true': output_for_true,
}

# Write the meta data to a YAML file
meta_data_file = f"{output_dir}/meta_data.yaml"
with open(meta_data_file, 'w') as file:
    yaml.dump(meta_data, file)